# 35 · CCC — descriptive ligand–receptor map, malignant CD4 ↔ TME

Runs off `data/ccc/ccc_skin.h5ad` (built and gated in **nb34**), ~727k skin cells × ~1,832
consensus-resource genes. Interactive; no LSF job needed.

This notebook is the **exploratory record** — every result frame lands in `tables/ccc_*.csv`.
The figures for reading are re-derived from scratch in **nb36**.

## Design

- Malignancy is **ALICE-TCR alone** (`mal_tcr_alice`), not `mal_combined`. The call is a
  TCR-sequence fact, so it is independent of the expression LIANA scores; inferCNV's is not.
  §13 keeps `mal_combined` / `mal_cnv` as sensitivity arms.
- Senders: `CD4_malignant` (76,349 cells, 29 donors) **and** `CD4_reactive` (72,207, 34 donors).
  The reactive comparator is what makes "malignant-specific" mean anything — it is the same
  lineage in the same lesion, so donor, study, chemistry and ambient RNA all affect it too.
  40,447 of the reactive cells do carry a `cnv_only` call; §13 tests that disagreement rather
  than assuming it away.
- Partners: CD8 · Myeloid · Fibroblast (core, 24–28 donors) then Keratinocyte · Vascular · B ·
  Plasma · Tregs (extended, 10–26 donors). Mast / Melanocyte are run but never claimed.
- Pooled over all 7 studies and all CTCL disease labels. **No stage, disease or layer
  stratification** — see `ccc_data.FORBIDDEN_CONTRASTS`: early-vs-advanced exists only inside
  li2024 (7 v 7 donors) and SS skin is buus2025 alone.

## How to read the numbers in here

LIANA's permutation p-value takes the **cell** as its unit. 76,349 malignant CD4 cells come from
29 donors, so those p-values are pseudoreplicated by roughly 2,600× and will approach 0 for
almost any pair. They size dots and mark cells; **they are not evidence**. What carries weight
here is: (i) the rank *delta* against the reactive comparator, (ii) stability across `expr_prop`,
(iii) survival of a within-donor label shuffle, and (iv) stability across the four malignancy
definitions. Donor-level inference is the deferred differential phase.

**Sections**: §1 load + coverage · §2 core axis · §3 extended · §4 thin · §5 malignant↔reactive ·
§6 comparator + rank delta · §7 controls · §8 expr_prop sweep · §9 within-donor shuffle ·
§10 downsample stability · §11 forced curated panel · §12 study diagnostic ·
§13 malignancy-definition sensitivity · §14 CD8 cnv-call sensitivity · §15 HC negative control.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc
import matplotlib as mpl, matplotlib.pyplot as plt
import plotnine as p9
import liana as li


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); sys.path.insert(0, str(NB_DIR))
import ccc_data as cd
import ccc_helpers as C

sc.settings.verbosity = 1
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["savefig.bbox"] = "tight"
cd.TAB_DIR.mkdir(parents=True, exist_ok=True); cd.FIG_DIR.mkdir(parents=True, exist_ok=True)
print("liana", li.__version__)
print("\n" + cd.CAVEAT_BLOCK)

## §1 · Load, resource, coverage, balanced subsample

In [ ]:
adata = C.load_ccc_adata()
adata.layers[cd.LAYER] = adata.X          # liana reads layer=LAYER; X already is lognorm
C.assert_ccc_invariants(adata)
resource, coverage = C.load_resource(var_names=adata.var_names)

In [ ]:
cov = C.coverage_table(adata, sample_key=cd.DONOR_KEY)
display(cov)
cov.to_csv(cd.TAB_DIR / "ccc_coverage_by_donor.csv")

In [ ]:
# Balanced subsample is the DEFAULT, for two reasons. Compute: 1000 permutations over ~400k
# cells is slow. Bias: liana's permutation p shrinks with group size, so an unbalanced
# CD4_malignant (76,349 cells vs Tregs' 1,936) scores p=0 on nearly every pair purely on n.
# The per-donor cap additionally stops li2024 (56% of skin cells) from dominating each level.
bal, sub_report = C.subsample_levels(adata)
sub_report.to_csv(cd.TAB_DIR / "ccc_subsample_report.csv", index=False)
print("\ndonors retained per level:")
print(bal.obs.groupby(bal.obs[cd.GROUPBY].astype(str), observed=True)[cd.DONOR_KEY]
      .nunique().sort_values(ascending=False).to_string())

## §2 · Core axis — malignant CD4 ↔ CD8 / Myeloid / Fibroblast

The primary run: 24–28 donors clear `MIN_CELLS` on both sides. Both directions are computed
in one call, and the dotplot panels by **sender**.

In [ ]:
sub_core, pairs_core, spec_core = C.prepare_analysis(bal, "ctcl_all_core")
res_core = C.run_rank_aggregate(sub_core, resource, groupby_pairs=pairs_core)
res_core.to_csv(cd.TAB_DIR / "ccc_liana_ctcl_all_core.csv", index=False)
print(res_core.shape)
display(res_core.sort_values("magnitude_rank").head(20))

In [ ]:
C.dotplot_axis(liana_res=res_core, source_labels=[cd.CD4_MALIGNANT],
               title="malignant CD4 -> core TME")

In [ ]:
C.dotplot_axis(liana_res=res_core, target_labels=[cd.CD4_MALIGNANT],
               title="core TME -> malignant CD4")

## §3 · Extended partners — keratinocyte, vascular, B, plasma, Tregs

In [ ]:
sub_ext, pairs_ext, spec_ext = C.prepare_analysis(bal, "ctcl_all_extended")
res_ext = C.run_rank_aggregate(sub_ext, resource, groupby_pairs=pairs_ext, key_added="liana_ext")
res_ext.to_csv(cd.TAB_DIR / "ccc_liana_ctcl_all_extended.csv", index=False)
C.dotplot_axis(liana_res=res_ext, source_labels=[cd.CD4_MALIGNANT],
               title="malignant CD4 -> extended TME")

In [ ]:
C.dotplot_axis(liana_res=res_ext, target_labels=[cd.CD4_MALIGNANT],
               title="extended TME -> malignant CD4")

## §4 · Thin partners — mast, melanocyte

Run so that absence is on the record rather than unexamined. Mast clears `MIN_CELLS` alongside
malignant CD4 in **4 donors** and melanocyte in 13; `axis_feasibility` marks both
`report_only`, and neither appears in nb36.

In [ ]:
sub_thin, pairs_thin, _ = C.prepare_analysis(bal, "ctcl_all_thin")
res_thin = C.run_rank_aggregate(sub_thin, resource, groupby_pairs=pairs_thin, key_added="liana_thin")
res_thin.to_csv(cd.TAB_DIR / "ccc_liana_ctcl_all_thin.csv", index=False)
display(res_thin.sort_values("magnitude_rank").head(10))

## §5 · Malignant CD4 ↔ reactive CD4

29 donors. Sender and receiver share a lineage, so shared-gene and autocrine artifacts are at
their maximum here: a pair whose ligand is expressed at similar proportion in *both* levels is
a CD4 property, not a directional interaction. §11 prints those proportions.

Both levels are now defined by the same TCR measurement — malignant is "in the ALICE clone
family", reactive is "tested and not in it" — so the axis no longer mixes a CNV-derived call on
one side with its absence on the other.

In [ ]:
sub_rea, pairs_rea, spec_rea = C.prepare_analysis(bal, "ctcl_all_reactive")
res_rea = C.run_rank_aggregate(sub_rea, resource, groupby_pairs=pairs_rea, key_added="liana_rea")
res_rea.to_csv(cd.TAB_DIR / "ccc_liana_ctcl_all_reactive.csv", index=False)
C.dotplot_axis(liana_res=res_rea, title="malignant CD4 <-> reactive CD4")

## §6 · The comparator, and the rank-delta table

Same partners, **reactive** CD4 as the sender. No claim of the form *"malignant CD4 signals X to
Y"* is reportable unless the pair is absent or clearly weaker on this axis — otherwise it is a
statement about CD4 T cells in inflamed skin.

Because the atlas has no usable normal-skin T cells (HC skin: 1,524 CD4 across 9 donors), this
is the only available baseline, and it is a *lesional* reactive population. So the claim this
supports is narrower and better defined: **"differs from reactive CD4 in the same lesion."**

In [ ]:
sub_cmp, pairs_cmp, spec_cmp = C.prepare_analysis(bal, "ctcl_all_comparator")
res_cmp = C.run_rank_aggregate(sub_cmp, resource, groupby_pairs=pairs_cmp, key_added="liana_cmp")
res_cmp.to_csv(cd.TAB_DIR / "ccc_liana_ctcl_all_comparator.csv", index=False)

sub_cmpx, pairs_cmpx, _ = C.prepare_analysis(bal, "ctcl_all_comparator_extended")
res_cmpx = C.run_rank_aggregate(sub_cmpx, resource, groupby_pairs=pairs_cmpx, key_added="liana_cmpx")
res_cmpx.to_csv(cd.TAB_DIR / "ccc_liana_ctcl_all_comparator_extended.csv", index=False)
print(res_cmp.shape, res_cmpx.shape)

In [ ]:
mal_out = pd.concat([res_core, res_ext]).query("source == @cd.CD4_MALIGNANT")
rea_out = pd.concat([res_cmp, res_cmpx]).query("source == @cd.CD4_REACTIVE")
delta_out = C.rank_delta(mal_out, rea_out)
delta_out.to_csv(cd.TAB_DIR / "ccc_malignant_vs_reactive_rank_delta_outgoing.csv", index=False)

mal_in = pd.concat([res_core, res_ext]).query("target == @cd.CD4_MALIGNANT").rename(
    columns={"source": "target", "target": "source"})
rea_in = pd.concat([res_cmp, res_cmpx]).query("target == @cd.CD4_REACTIVE").rename(
    columns={"source": "target", "target": "source"})
delta_in = C.rank_delta(mal_in, rea_in)
delta_in.to_csv(cd.TAB_DIR / "ccc_malignant_vs_reactive_rank_delta_incoming.csv", index=False)

print("outgoing -- most malignant-shifted (negative delta = better rank from malignant CD4):")
display(delta_out.head(25))
print("\nfound only with malignant CD4 as sender:", int(delta_out['malignant_only'].sum()))

In [ ]:
print("incoming -- TME pairs most shifted toward malignant CD4 as the receiver:")
display(delta_in.head(25))

## §7 · Controls

Positives are drawn from `docs/CTCL_atlas_notebook_replication_spec.md` (Fig 5f/6f/8a) and the
CTCL literature; the three the spec names explicitly (`CXCL13–CXCR5`, `CD40LG–CD40`,
`CD86–CD28` on the malignant↔B axis) are held to a stricter bar. Negatives are
lineage-impossible edges — a T cell does not send `KITLG` or `COL1A1`.

`MIF–CD74` sitting at the top is the **documented failure mode**, not a finding: MIF is
ubiquitous and very highly expressed, so it is the positive control for the null model breaking.

In [ ]:
res_all = pd.concat([res_core, res_ext, res_rea], ignore_index=True)
ctrl, caveat_hits = C.control_report(res_all, resource=resource, top_n=40)
ctrl.to_csv(cd.TAB_DIR / "ccc_control_report.csv", index=False)
caveat_hits.to_csv(cd.TAB_DIR / "ccc_caveat_hits.csv", index=False)
display(ctrl)

In [ ]:
spec_ctrl = C.resolve_controls(cd.SPEC_MUST_HAVES, resource)
found = ctrl.merge(pd.DataFrame(spec_ctrl, columns=["source", "target", "ligand_complex", "receptor_complex"]),
                   on=["source", "target", "ligand_complex", "receptor_complex"])
print("replication-spec must-haves:")
display(found)
if len(found) and "pct_of_tested" in found:
    print("in the top quartile of magnitude_rank:", (found['pct_of_tested'] < 0.25).sum(), "/", len(found))

## §8 · `expr_prop` sensitivity

`expr_prop` is the single biggest lever on the result, and the low-abundance cytokines
(`IL13`, `IL4`, `IL31`, `IL2`) sit right at the threshold. Anything not stable across
0.05 / 0.1 / 0.2 is a threshold artifact.

In [ ]:
sweep, stable = C.expr_prop_sweep(sub_core, resource, groupby_pairs=pairs_core, verbose=False)
sweep.to_csv(cd.TAB_DIR / "ccc_expr_prop_sweep_core.csv", index=False)
print(f"{len(stable)}/{cd.TOP_N} of the top-{cd.TOP_N} are stable across expr_prop "
      f"{cd.EXPR_PROP_SWEEP}")
display(pd.DataFrame(sorted(stable), columns=["source", "target", "ligand_complex", "receptor_complex"]))

## §9 · Within-donor label shuffle

The grouping label is permuted **within each donor**, so every donor's cell-type composition is
held fixed and what dissolves is cell-type specificity rather than donor identity. A global
shuffle would also destroy composition and pass trivially.

Target: **≤ 2/20** overlap with the real top-20.

In [ ]:
shuf, overlap = C.shuffle_control(sub_core, resource, groupby_pairs=pairs_core,
                                  reference=res_core, verbose=False)
overlap.to_csv(cd.TAB_DIR / "ccc_shuffle_control.csv", index=False)
print("\nmean overlap:", overlap['overlap_with_real'].mean(),
      "-- target <= 2 of", cd.TOP_N)

## §10 · Downsample stability

Makes the n-dependence of the permutation p-value explicit instead of leaving it as an unstated
property of the default subsample. Watch `frac_pval_zero` grow with `n_cells`: that is the
pseudoreplication, measured.

In [ ]:
stab, tops = C.downsample_stability(sub_core, resource, groupby_pairs=pairs_core)
stab.to_csv(cd.TAB_DIR / "ccc_downsample_stability.csv", index=False)
display(stab)

## §11 · Forced curated panel

Every curated pair is reported whether or not it cleared `expr_prop`, with the actual detection
proportion in sender and receiver. A pair that failed is then reported as *"ligand detected in
6 % of the sender"* rather than as absent — a coverage limit must never be read as biology.

This is also where the malignant↔reactive autocrine caveat from §5 is checkable: compare the
ligand's `expr_prop` in `CD4_malignant` against `CD4_reactive`.

In [ ]:
for gene, note in cd.LR_CAVEATS.items():
    print(f"{gene}: {note}\n")

In [ ]:
panel_res = pd.concat(
    [C.filter_to_panel(r, resource=resource).assign(analysis=name)
     for name, r in [("core", res_core), ("extended", res_ext), ("reactive", res_rea),
                     ("comparator", res_cmp), ("comparator_extended", res_cmpx)]],
    ignore_index=True,
)
panel_res.to_csv(cd.TAB_DIR / "ccc_liana_curated_panel.csv", index=False)
print(f"{len(panel_res)} scored rows over {panel_res.group.nunique()} panel groups")
display(panel_res.groupby("group").size().rename("n_scored").to_frame())

In [ ]:
fpe = C.forced_panel_expression(bal, resource=resource)
fpe.to_csv(cd.TAB_DIR / "ccc_forced_panel_expression.csv", index=False)

# which curated genes sit just under the expr_prop threshold in the senders -- these are the
# pairs whose absence from the result is a detection limit, not a negative
focus = fpe[fpe.level.isin([cd.CD4_MALIGNANT, cd.CD4_REACTIVE, "Myeloid", "Fibroblast", "B"])]
near = focus[(focus.expr_prop > 0.01) & (focus.expr_prop < cd.EXPR_PROP)]
print(f"curated genes between 1% and expr_prop={cd.EXPR_PROP} in a focal level:")
display(near.sort_values("expr_prop", ascending=False).head(30))

In [ ]:
# autocrine check for the malignant <-> reactive axis: a ligand at similar proportion in both
# CD4 levels is a CD4 property, not a directional signal
piv = (focus[focus.level.isin([cd.CD4_MALIGNANT, cd.CD4_REACTIVE])]
       .pivot(index="gene", columns="level", values="expr_prop"))
piv["ratio_mal_over_rea"] = piv[cd.CD4_MALIGNANT] / piv[cd.CD4_REACTIVE].replace(0, np.nan)
print("curated genes most malignant-skewed within CD4:")
display(piv.sort_values("ratio_mal_over_rea", ascending=False).head(20))
print("curated genes symmetric between the two CD4 levels (autocrine artifact risk):")
display(piv[(piv.ratio_mal_over_rea.between(0.8, 1.25)) & (piv[cd.CD4_MALIGNANT] > cd.EXPR_PROP)]
        .head(20))

## §12 · Study diagnostic

`expr_prop` is a detection-rate threshold and chemistry tracks study (5′ / 3′ / FFPE Flex), so
a pair that passes in one study and fails in another is a capture artifact rather than a
biological difference. Only studies with enough donors on both sides of the core axis are run.

In [ ]:
ok_studies = [
    s for s in bal.obs[cd.STUDY_KEY].astype(str).unique()
    if (bal.obs[bal.obs[cd.STUDY_KEY].astype(str) == s]
        .groupby(cd.GROUPBY, observed=True)[cd.DONOR_KEY].nunique()
        .reindex([cd.CD4_MALIGNANT, "Myeloid"]).fillna(0) >= 3).all()
]
print("studies with >= 3 donors on both sides:", ok_studies)
by_study = C.run_by_group(sub_core, resource, cd.STUDY_KEY, groups=ok_studies,
                          groupby_pairs=pairs_core, verbose=False)
by_study.to_csv(cd.TAB_DIR / "ccc_liana_by_study_core.csv", index=False)

In [ ]:
if len(by_study):
    named = {s: by_study[by_study[cd.STUDY_KEY] == s] for s in ok_studies}
    ov = C.top_n_overlap_matrix(named)
    print(f"top-{cd.TOP_N} overlap between studies (core axis):")
    display(ov)
    ov.to_csv(cd.TAB_DIR / "ccc_study_topn_overlap.csv")
    keys = ["source", "target", "ligand_complex", "receptor_complex"]
    per = by_study.groupby(keys)[cd.STUDY_KEY].nunique().rename("n_studies")
    top = res_core.sort_values("magnitude_rank").head(30).set_index(keys)
    print("\nhow many studies recover each of the top-30 pooled pairs:")
    display(top.join(per).reset_index()[keys + ["magnitude_rank", "n_studies"]])

## §13 · Malignancy-definition sensitivity

The primary call is **ALICE-TCR alone**, chosen because a TCR-sequence fact is independent of
the expression LIANA scores. The alternatives are not: `mal_cnv` is inferCNV's per-donor cluster
call, derived from *smoothed regional expression*, so a `cnv_only` label and that cell's
expression are partly the same measurement — a pair whose ligand or receptor sits on a
recurrently gained arm could be reporting dosage. `mal_combined` is `mal_cnv ∪ mal_tcr_alice`
and inherits the problem for the 77,624 CD4 it adds on top of ALICE.

This section runs the core axis under all four definitions anyway. The point is no longer to
pick one — it is to show the headline pairs do not depend on the choice. Two things to read:

- **ALICE vs mal_cnv** is the load-bearing comparison. They disagree on 40,447 CD4 that ALICE
  calls reactive and inferCNV calls malignant; if the top pairs survive both, the circularity
  never mattered.
- Any headline pair not stable across **≥3 of the 4 definitions** is demoted.

The CDR3 gate that separates reactive from unassessed is applied to the ALICE primary **only**
(`build_ccc_celltype_from` passes `tcr_assessed_src=None`): "could ALICE test this cell" is
meaningless for a CNV- or paper-label-derived call, and gating those on CDR3 recovery would mix
unrelated missingness mechanisms and make the overlap matrix uninterpretable.

In [ ]:
alt_res, alt_counts = {cd.MALIG_SRC: res_core}, {}
for defn in ["mal_combined", "mal_cnv", "tumor_cell"]:
    a = bal.copy()
    a.obs[cd.GROUPBY] = C.build_ccc_celltype_from(a.obs, defn, verbose=False)
    a = a[a.obs[cd.GROUPBY].notna()].copy()
    a.layers[cd.LAYER] = a.X
    alt_counts[defn] = a.obs[cd.GROUPBY].value_counts().to_dict()
    try:
        alt_res[defn] = C.run_rank_aggregate(a, resource, groupby_pairs=pairs_core,
                                             key_added=f"liana_{defn}", verbose=False)
    except Exception as exc:
        print(f"{defn}: {exc}")
print(f"primary = {cd.MALIG_SRC}; alternatives relabelled without the CDR3 gate\n")
print(pd.DataFrame(alt_counts).fillna(0).astype(int).to_string())

In [ ]:
ov_def = C.top_n_overlap_matrix(alt_res)
print(f"top-{cd.TOP_N} overlap across malignancy definitions (core axis):")
display(ov_def)
ov_def.to_csv(cd.TAB_DIR / "ccc_malignancy_definition_topn_overlap.csv")

keys = ["source", "target", "ligand_complex", "receptor_complex"]
n_def = (pd.concat([r.sort_values("magnitude_rank").head(cd.TOP_N).assign(defn=k)
                    for k, r in alt_res.items()])
         .groupby(keys)["defn"].nunique().rename("n_definitions"))
n_def.sort_values(ascending=False).to_csv(cd.TAB_DIR / "ccc_pair_definition_stability.csv")
print("\npairs in the top-20 of all four definitions:")
display(n_def[n_def == len(alt_res)].to_frame())

In [ ]:
# the evidence-strength view: restrict malignant CD4 to the 62,096 cells that ALSO carry a CNV
# call. Under an ALICE primary this is a NESTED subset of CD4_malignant (62,096 of 76,349), not
# an orthogonal definition -- it asks whether the 14,253 tcr_only cells drive anything.
both = bal[(bal.obs[cd.GROUPBY].astype(str) != cd.CD4_MALIGNANT)
           | (bal.obs[cd.EVIDENCE_SRC].astype(str) == "both")].copy()
both.layers[cd.LAYER] = both.X

is_mal = bal.obs[cd.GROUPBY].astype(str) == cd.CD4_MALIGNANT
ev_both = bal.obs[cd.EVIDENCE_SRC].astype(str) == "both"
assert not (ev_both & ~is_mal & (bal.obs[cd.CELLTYPE_SRC].astype(str) == "CD4")).any(), \
    "evidence=='both' CD4 outside CD4_malignant -- 'both' should be nested inside ALICE"
print(f"malignant CD4 in the subsample: {int(is_mal.sum())} "
      f"-> restricted to evidence=='both': {int((is_mal & ev_both).sum())}")
print("donors:", both.obs.loc[both.obs[cd.GROUPBY].astype(str) == cd.CD4_MALIGNANT,
                              cd.DONOR_KEY].nunique())
res_both = C.run_rank_aggregate(both, resource, groupby_pairs=pairs_core,
                                key_added="liana_both", verbose=False)
res_both.to_csv(cd.TAB_DIR / "ccc_liana_core_evidence_both.csv", index=False)
display(C.top_n_overlap_matrix({cd.MALIG_SRC: res_core, "evidence_both": res_both}))

## §14 · CD8 sensitivity — drop the cnv-only malignant calls

ALICE is a CD4-only caller, so under the primary definition **no CD8 cell is malignant** and the
CD8 level is a single pooled level by construction rather than by choice. That does not make the
question go away: `mal_cnv` flags 21,200 of 51,831 skin CD8 as malignant, all `cnv_only` with
zero TCR support — a genuine CD8 malignancy, a CNV false positive, or ambient bleed from the
dominant CD4 clone. If those cells are transcriptionally odd, the CD8 comparator level is
contaminated whichever call is primary.

So the drop is driven off `mal_combined` explicitly (`build_ccc_celltype` with the primary
would drop nothing), and this run quantifies what the contamination costs.

In [ ]:
cd8_alt = bal.copy()
# CD4 levels stay on the ALICE primary; only the CD8 drop reads mal_combined, because that is
# where the cnv_only CD8 flag lives. Build the primary label, then null out those CD8 cells.
cd8_flagged = ((cd8_alt.obs[cd.CELLTYPE_SRC].astype(str) == "CD8")
               & cd8_alt.obs["mal_combined"].fillna(False).astype(bool))
lab = C.build_ccc_celltype(cd8_alt.obs, verbose=False)
cd8_alt.obs[cd.GROUPBY] = lab.mask(cd8_flagged)      # mask -> NaN, categories preserved
print(f"dropped {int(cd8_flagged.sum())} CD8 cells with a cnv_only malignant call")
cd8_alt = cd8_alt[cd8_alt.obs[cd.GROUPBY].notna()].copy()
cd8_alt.obs[cd.GROUPBY] = cd8_alt.obs[cd.GROUPBY].cat.remove_unused_categories()
cd8_alt.layers[cd.LAYER] = cd8_alt.X
res_cd8 = C.run_rank_aggregate(
    cd8_alt, resource,
    groupby_pairs=C.build_groupby_pairs({"ax": ([cd.CD4_MALIGNANT], ["CD8"])}),
    key_added="liana_cd8_clean", verbose=False)
res_cd8.to_csv(cd.TAB_DIR / "ccc_liana_cd8_drop_cnvonly.csv", index=False)

cd8_ref = res_core[res_core[["source", "target"]].isin([cd.CD4_MALIGNANT, "CD8"]).all(axis=1)]
display(C.top_n_overlap_matrix({"cd8_as_is": cd8_ref, "cd8_cnvonly_dropped": res_cd8}))

## §15 · Healthy-skin negative control

HC skin has 27,534 cells but only 1,524 CD4 and 300 CD8 across 9 donors, so **no T-cell axis is
computable** — `CD4_malignant` must be 0 cells here, and that is the expected result rather than
a failure. What is computable is the structural fibroblast → keratinocyte / vascular / myeloid
grid, which should recover plausible matrix and growth-factor edges. If it does not, the problem
is the object, not the biology.

In [ ]:
hc, pairs_hc, spec_hc = C.prepare_analysis(adata, "hc_structural")
n_mal = int((hc.obs[cd.GROUPBY].astype(str) == cd.CD4_MALIGNANT).sum())
print(f"\nmalignant CD4 in HC skin: {n_mal} (expected 0)")
assert n_mal == 0, "a malignant CD4 call in healthy skin needs explaining before going further"

hc_bal, _ = C.subsample_levels(hc, verbose=False)
res_hc = C.run_rank_aggregate(hc_bal, resource, groupby_pairs=pairs_hc,
                              key_added="liana_hc", verbose=False)
res_hc.to_csv(cd.TAB_DIR / "ccc_liana_hc_structural.csv", index=False)
display(res_hc.sort_values("magnitude_rank").head(20))

### Outcome

Result frames are in `tables/ccc_*.csv`. Read the reportable set as the intersection of:
top-ranked in §2/§3 · shifted against the reactive comparator in §6 · present in the control
report §7 · stable across `expr_prop` §8 · dissolved by the within-donor shuffle §9 · stable
across ≥3 malignancy definitions §13.

Headline figures: **`36_ccc_headline_figures.ipynb`**.